# 04 – Cross-Exchange Comparison

Compare findings across Binance, OKX, Bybit, Coinbase, Kraken.
Key question: do higher-volume exchanges show weaker multifractality?

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, '..')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)
tables_dir = Path('../results/tables')

## Load summary tables

In [ ]:
def load(name):
    p = tables_dir / f'{name}.csv'
    return pd.read_csv(p) if p.exists() else None
pv = load('p_variation_summary')
mf = load('mfdfa_summary')
ms = load('moment_scaling_summary')
wl = load('wavelet_leaders_summary')
for name, df in [('p_variation', pv), ('mfdfa', mf), ('moment_scaling', ms), ('wavelet', wl)]:
    if df is not None: print(f'{name}: {df.shape}')

## Main finding: is log W strictly negative everywhere?

In [ ]:
if pv is not None:
    print('Fraction of series with no valid H estimate:')
    print(pv['H_standard'].isna().mean())
    print()
    print('Min/max log W across all series:')
    print(pv[['log_W_min_std', 'log_W_max_std']].describe())

## Spectral width by exchange

In [ ]:
if mf is not None:
    sw = mf.groupby(['exchange', 'frequency'])['spectral_width'].mean().unstack()
    display(sw)
    sw.plot(kind='bar', figsize=(9, 4))
    plt.ylabel('Mean MF-DFA spectral width')
    plt.title('Multifractality by exchange')
    plt.legend(title='Frequency')
    plt.tight_layout()

## BTC vs ETH

In [ ]:
if mf is not None:
    for asset in mf['asset'].unique():
        sub = mf[mf['asset'] == asset]
        print(f"\n{asset}: mean spectral width = {sub['spectral_width'].mean():.4f} (+/- {sub['spectral_width'].std():.4f})")

## Roughness index H (where valid)

In [ ]:
if pv is not None:
    h_valid = pv.dropna(subset=['H_standard'])
    if h_valid.empty:
        print('No valid roughness index found — log W remains strictly negative.')
        print('This confirms the main finding of Pontiggia (2025) extends across exchanges.')
    else:
        print(f'{len(h_valid)} series with valid H estimate:')
        display(h_valid[['exchange', 'asset', 'frequency', 'year', 'H_standard', 'log_W_min_std', 'log_W_max_std']])